# Chapter 15 — Ensemble Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch15_ensemble_methods/ch15_notebook.ipynb)

Companion notebook for **Chapter 15** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- A group of weak models, combined properly, reliably beats any one of them alone.
- Bagging trains many models in parallel on bootstrap samples and averages them, which mainly reduces variance.
- Random Forest adds a second source of randomness by considering only a subset of features at each split.
- Boosting (AdaBoost) works sequentially, giving more weight to the examples the previous models got wrong.
- Random Forest also hands you feature importance scores for free, useful for feature selection.

## Example 1: Bagging — Weak Classifiers Growing Stronger

This example shows the magic of Bagging: taking individually weak shallow trees (depth 2) and combining 10 to 200 of them into a stronger ensemble. A single shallow tree gets 75.3%, but 50 bagged ones push to 84.7%. The Random Forest takes this further by also randomizing which features each tree considers at each split — it achieves 91.3%. Notice also that a single deep (unlimited) tree gets 84.0% — similar to the bagged shallow trees but with more risk of overfitting. The visual bar chart makes the improvement instantly obvious.

In [1]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                            n_redundant=5, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"{'Model':<38} {'Test Acc':>10}")
print("-" * 50)
results = {}

# Single trees
for name, clf in [('Single Tree (deep)', DecisionTreeClassifier(random_state=42)),
                   ('Single Shallow Tree (depth=2)', DecisionTreeClassifier(max_depth=2, random_state=42))]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    results[name] = acc

# Bagging
for L in [10, 50, 100, 200]:
    bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                            n_estimators=L, random_state=42, bootstrap=True)
    bag.fit(X_train, y_train)
    results[f'Bagging {L:>3} shallow trees'] = accuracy_score(y_test, bag.predict(X_test))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results['Random Forest (100 trees)'] = accuracy_score(y_test, rf.predict(X_test))

for name, acc in results.items():
    bar = 'X' * int(acc * 40)
    print(f"  {name:<36} {acc*100:>6.1f}%  |{bar}")

print()
print("Key insight: Weak classifiers (shallow trees) become strong when bagged together.")
print("Random Forest adds feature randomness on top of Bagging for even better results.")

Model                                    Test Acc
--------------------------------------------------
  Single Tree (deep)                     84.0%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Single Shallow Tree (depth=2)          75.3%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Bagging  10 shallow trees              81.3%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Bagging  50 shallow trees              84.7%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Bagging 100 shallow trees              82.7%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Bagging 200 shallow trees              82.0%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  Random Forest (100 trees)              91.3%  |XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Key insight: Weak classifiers (shallow trees) become strong when bagged together.
Random Forest adds feature randomness on top of Bagging for even better results.


## Example 2: AdaBoost — Progressive Accuracy Improvement

AdaBoost is sequential Bagging's smarter cousin. Instead of building classifiers in parallel on random subsets, it builds them one after another, always paying extra attention to the examples that were misclassified so far. We track how the ensemble accuracy grows as we add more weak classifiers. A single stump starts at 70%. By round 50 it reaches 85% — a 15 percentage point gain purely from sequentially reweighting examples. The final comparison shows AdaBoost vs Random Forest vs a single tree: the random forest clearly beats the single tree, while AdaBoost with 200 stumps only matches it, because its accuracy dips slightly after round 50. More rounds are not always better.

In [2]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

np.random.seed(0)
X, y = make_classification(n_samples=400, n_features=10, n_informative=6,
                            n_classes=2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

print("=== AdaBoost: Accuracy vs Number of Weak Classifiers ===\n")
print(f"{'Classifiers':>12}  {'Ensemble Acc':>14}  {'vs Baseline'}")
print("-" * 45)

single_acc = None
for n in [1, 2, 5, 10, 20, 50, 100, 200]:
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                             n_estimators=n, random_state=0)
    ada.fit(X_train, y_train)
    acc = accuracy_score(y_test, ada.predict(X_test))
    if n == 1: single_acc = acc
    diff = f"+{(acc-single_acc)*100:.1f}%" if n > 1 else "(baseline)"
    print(f"{n:>12}  {acc*100:>13.1f}%  {diff}")

print(f"\nBaseline (1 stump): {single_acc*100:.1f}%")
print()
print("=== Final Model Comparison ===\n")
for name, model in [
    ('Single Tree (depth=5)',    DecisionTreeClassifier(max_depth=5, random_state=0)),
    ('Random Forest 100 trees',  RandomForestClassifier(n_estimators=100, random_state=0)),
    ('AdaBoost 200 stumps',      AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                                                     n_estimators=200, random_state=0)),
]:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"  {name:<28}: {acc*100:.1f}%")

=== AdaBoost: Accuracy vs Number of Weak Classifiers ===

 Classifiers    Ensemble Acc  vs Baseline
---------------------------------------------
           1           70.0%  (baseline)
           2           70.0%  +0.0%
           5           71.7%  +1.7%
          10           78.3%  +8.3%
          20           80.8%  +10.8%
          50           85.0%  +15.0%
         100           82.5%  +12.5%
         200           83.3%  +13.3%

Baseline (1 stump): 70.0%

=== Final Model Comparison ===

  Single Tree (depth=5)       : 83.3%
  Random Forest 100 trees     : 88.3%
  AdaBoost 200 stumps         : 83.3%


## Example 3: Random Forest Feature Importance

One powerful bonus of Random Forest is that it gives you feature importance scores for free. After training 100 trees, each tree tells us how much each feature contributed to reducing impurity across all its splits. We average this across all trees to get a robust importance ranking. On the Breast Cancer dataset (30 features, 569 samples), the model achieves 97.08% accuracy. The top features (mean concave points, worst concave points, worst area) are clinically meaningful — they measure tumor shape irregularity. The bottom features (symmetry error, texture error) contribute almost nothing and could safely be dropped.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc = accuracy_score(y_test, rf.predict(X_test))

print(f"Dataset: Breast Cancer | {X.shape[0]} samples, {X.shape[1]} features")
print(f"Test Accuracy: {acc*100:.2f}%\n")
print("Top 10 Most Important Features:")
print(f"  {'Rank':<5} {'Feature':<30} {'Importance':>11}")
print("-" * 50)

ranked = sorted(zip(rf.feature_importances_, data.feature_names), reverse=True)
for rank, (imp, name) in enumerate(ranked[:10], 1):
    bar = 'X' * int(imp * 150)
    print(f"  {rank:<5} {name:<30} {imp:>10.4f}  {bar}")

print()
print("Bottom 5 (candidates to drop):")
for imp, name in ranked[-5:]:
    print(f"  {name:<30}: {imp:.4f}")

Dataset: Breast Cancer | 569 samples, 30 features
Test Accuracy: 97.08%

Top 10 Most Important Features:
  Rank  Feature                         Importance
--------------------------------------------------
  1     mean concave points                0.1419  XXXXXXXXXXXXXXXXXXXXX
  2     worst concave points               0.1271  XXXXXXXXXXXXXXXXXXX
  3     worst area                         0.1182  XXXXXXXXXXXXXXXXX
  4     mean concavity                     0.0806  XXXXXXXXXXXX
  5     worst radius                       0.0780  XXXXXXXXXXX
  6     worst perimeter                    0.0743  XXXXXXXXXXX
  7     mean perimeter                     0.0601  XXXXXXXXX
  8     mean area                          0.0538  XXXXXXXX
  9     worst concavity                    0.0411  XXXXXX
  10    mean radius                        0.0323  XXXX

Bottom 5 (candidates to drop):
  symmetry error                : 0.0040
  concave points error          : 0.0034
  mean symmetry                 : 0.0033
